In [3]:
import optuna
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [7]:
url = "https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/master/diabetes.csv"
columns = ['Pregnancies','Glucose', 'BloodPressure', 'SkinThickness','Insulin', 'BMI','DiabetesPedigreeFunction','Age', 'Outcome']

df = pd.read_csv(url, names=columns, header=0)

In [8]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [9]:
import numpy as np

In [13]:
cols_with_missing_values = ['Glucose', 'BloodPressure', 'SkinThickness','Insulin', 'BMI']
df[cols_with_missing_values] = df[cols_with_missing_values].replace(0,np.nan)
df.fillna(df.mean(), inplace=True)
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148.0,72.0,35.00000,155.548223,33.6,0.627,50,1
1,1,85.0,66.0,29.00000,155.548223,26.6,0.351,31,0
2,8,183.0,64.0,29.15342,155.548223,23.3,0.672,32,1
3,1,89.0,66.0,23.00000,94.000000,28.1,0.167,21,0
4,0,137.0,40.0,35.00000,168.000000,43.1,2.288,33,1


In [14]:
X = df.iloc[:,:8]
y = df['Outcome']

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X,y,train_size=0.2,random_state=28)

In [16]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

def objective(trial):
    n_estimators = trial.suggest_int('n_estimators',50,200)
    max_depth = trial.suggest_int('max_depth',3,20)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=28
    )

    score = cross_val_score(model , X_train, y_train,cv=5,scoring='accuracy').mean()

    return score

In [25]:
import optuna.samplers


study = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler())
study.optimize(objective,n_trials = 100)

[I 2025-06-07 15:14:30,919] A new study created in memory with name: no-name-f58666f1-7a25-4a27-84ae-8ab4e9a2b35d
[I 2025-06-07 15:14:31,332] Trial 0 finished with value: 0.7580645161290323 and parameters: {'n_estimators': 77, 'max_depth': 3}. Best is trial 0 with value: 0.7580645161290323.
[I 2025-06-07 15:14:32,301] Trial 1 finished with value: 0.7716129032258066 and parameters: {'n_estimators': 197, 'max_depth': 19}. Best is trial 1 with value: 0.7716129032258066.
[I 2025-06-07 15:14:32,827] Trial 2 finished with value: 0.7651612903225806 and parameters: {'n_estimators': 104, 'max_depth': 14}. Best is trial 1 with value: 0.7716129032258066.
[I 2025-06-07 15:14:33,275] Trial 3 finished with value: 0.7649462365591397 and parameters: {'n_estimators': 93, 'max_depth': 12}. Best is trial 1 with value: 0.7716129032258066.
[I 2025-06-07 15:14:33,557] Trial 4 finished with value: 0.7449462365591397 and parameters: {'n_estimators': 61, 'max_depth': 4}. Best is trial 1 with value: 0.771612903

In [20]:
print(study.best_trial.value)
print(study.best_trial.params)

0.7782795698924732
{'n_estimators': 56, 'max_depth': 16}


In [21]:
from sklearn.metrics import accuracy_score


best_model = RandomForestClassifier(n_estimators=56,max_depth=16)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.7642276422764228


In [22]:
study = optuna.create_study(direction='maximize',sampler=optuna.samplers.RandomSampler())
study.optimize(objective,n_trials = 50)

[I 2025-06-07 15:11:36,024] A new study created in memory with name: no-name-1816c1d2-fa85-42bf-a2ad-1568e4eedb7d
[I 2025-06-07 15:11:36,973] Trial 0 finished with value: 0.732258064516129 and parameters: {'n_estimators': 197, 'max_depth': 5}. Best is trial 0 with value: 0.732258064516129.
[I 2025-06-07 15:11:37,313] Trial 1 finished with value: 0.732258064516129 and parameters: {'n_estimators': 73, 'max_depth': 5}. Best is trial 0 with value: 0.732258064516129.
[I 2025-06-07 15:11:38,192] Trial 2 finished with value: 0.7716129032258066 and parameters: {'n_estimators': 192, 'max_depth': 15}. Best is trial 2 with value: 0.7716129032258066.
[I 2025-06-07 15:11:38,823] Trial 3 finished with value: 0.7716129032258064 and parameters: {'n_estimators': 136, 'max_depth': 17}. Best is trial 2 with value: 0.7716129032258066.
[I 2025-06-07 15:11:39,477] Trial 4 finished with value: 0.7453763440860215 and parameters: {'n_estimators': 142, 'max_depth': 6}. Best is trial 2 with value: 0.771612903225

In [23]:
search_space = {
    'n_estimators' : [50,100,150,200],
    'max_depth' : [5,10,15,20]
}

In [24]:
study = optuna.create_study(direction='maximize',sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2025-06-07 15:13:50,367] A new study created in memory with name: no-name-a685908c-6ec7-446f-a225-fa7557ebf81a
[I 2025-06-07 15:13:50,842] Trial 0 finished with value: 0.7255913978494624 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7255913978494624.
[I 2025-06-07 15:13:51,569] Trial 1 finished with value: 0.7713978494623657 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 1 with value: 0.7713978494623657.
[I 2025-06-07 15:13:51,848] Trial 2 finished with value: 0.7718279569892473 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7718279569892473.
[I 2025-06-07 15:13:52,453] Trial 3 finished with value: 0.7716129032258064 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7718279569892473.
[I 2025-06-07 15:13:53,007] Trial 4 finished with value: 0.7716129032258064 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.771827

In [26]:
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [27]:
plot_optimization_history(study).show()

In [28]:
plot_parallel_coordinate(study).show()

In [29]:
plot_slice(study).show()

In [30]:
plot_contour(study).show()

In [31]:
plot_param_importances(study)

In [32]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC

In [33]:
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [34]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2025-06-07 15:26:20,600] A new study created in memory with name: no-name-940e9acd-309f-4ca5-82a9-937598f49ad8
[I 2025-06-07 15:26:21,259] Trial 0 finished with value: 0.7516339869281046 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 237, 'learning_rate': 0.0632473706734536, 'max_depth': 11, 'min_samples_split': 7, 'min_samples_leaf': 9}. Best is trial 0 with value: 0.7516339869281046.
[I 2025-06-07 15:26:21,486] Trial 1 finished with value: 0.7320261437908497 and parameters: {'classifier': 'RandomForest', 'n_estimators': 71, 'max_depth': 15, 'min_samples_split': 5, 'min_samples_leaf': 2, 'bootstrap': True}. Best is trial 0 with value: 0.7516339869281046.
[I 2025-06-07 15:26:21,495] Trial 2 finished with value: 0.6862745098039215 and parameters: {'classifier': 'SVM', 'C': 0.23718128704139455, 'kernel': 'rbf', 'gamma': 'scale'}. Best is trial 0 with value: 0.7516339869281046.
[I 2025-06-07 15:26:22,154] Trial 3 finished with value: 0.738562091503268 and parameters

In [35]:
best_trial = study.best_trial
print(best_trial.params)
print(best_trial.value)

{'classifier': 'SVM', 'C': 2.708605322232677, 'kernel': 'linear', 'gamma': 'auto'}
0.7777777777777777


In [37]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.751634,2025-06-07 15:26:20.601562,2025-06-07 15:26:21.259016,0 days 00:00:00.657454,NaN,NaN,GradientBoosting,NaN,NaN,0.063247,11.0,9.0,7.0,237.0,COMPLETE
1,1,0.732026,2025-06-07 15:26:21.259881,2025-06-07 15:26:21.485876,0 days 00:00:00.225995,NaN,True,RandomForest,NaN,NaN,NaN,15.0,2.0,5.0,71.0,COMPLETE
2,2,0.686275,2025-06-07 15:26:21.486596,2025-06-07 15:26:21.495213,0 days 00:00:00.008617,0.237181,NaN,SVM,scale,rbf,NaN,NaN,NaN,NaN,NaN,COMPLETE
3,3,0.738562,2025-06-07 15:26:21.495831,2025-06-07 15:26:22.154551,0 days 00:00:00.658720,NaN,True,RandomForest,NaN,NaN,NaN,14.0,10.0,4.0,224.0,COMPLETE
4,4,0.712418,2025-06-07 15:26:22.155323,2025-06-07 15:26:22.566577,0 days 00:00:00.411254,NaN,NaN,GradientBoosting,NaN,NaN,0.017675,3.0,9.0,2.0,205.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.751634,2025-06-07 15:26:31.979536,2025-06-07 15:26:31.992916,0 days 00:00:00.013380,1.733981,NaN,SVM,scale,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.777778,2025-06-07 15:26:31.993698,2025-06-07 15:26:32.007419,0 days 00:00:00.013721,1.210595,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.777778,2025-06-07 15:26:32.008013,2025-06-07 15:26:32.055865,0 days 00:00:00.047852,0.507339,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
98,98,0.777778,2025-06-07 15:26:32.056669,2025-06-07 15:26:32.076028,0 days 00:00:00.019359,1.513144,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [39]:
study.trials_dataframe()['params_classifier'].value_counts()

params_classifier
SVM                 80
GradientBoosting    10
RandomForest        10
Name: count, dtype: int64

In [40]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

params_classifier
GradientBoosting    0.733333
RandomForest        0.739869
SVM                 0.762745
Name: value, dtype: float64